## Proof with checker — primo esperimento

Il modello risponde a una query di sussunzione `Sub ⊑ Sup` restituendo un JSON con:

- `answer`: `"yes"` / `"no"`
- `justification`: lista di assiomi **Turtle** copiati dall'ontologia, **uno per elemento**; lista vuota se `answer == "no"`.

Un checker esterno verifica poi la justification:

1. **parse** di ogni assioma con rdflib (il checker aggiunge le righe `@prefix`);
2. **vocabolario**: le entità citate esistono in O;
3. **fedeltà**: ogni assioma è presente in O (confronto strutturale con rdflib, blank node inclusi);
4. **sufficienza**: HermiT, su un'ontologia che contiene **solo** J, verifica che J ⊨ query;
5. confronto con la **ground truth** (HermiT su O completa).

`certificate_valid = all_parsed ∧ all_in_O ∧ J ⊨ query`.

In [7]:
from ollama import Client
from ollama import ChatResponse
from typing import Optional, Any


def generate_ollama_ans(
    model: str,
    user_query: str,
    context: Optional[str] = None,
    system_prompt: Optional[str] = None,
    num_ctx: int = 8192,
    host: str = "http://localhost:11434",
    think: Optional[bool | str] = None,
    format: Optional[str | dict] = None,
    **kwargs: Any
) -> ChatResponse:
    """
    Generate a response using a local Ollama model.

    Args:
        model: Name of the Ollama model.
        user_query: Question to answer.
        context: Optional ontology or other reasoning context.
        system_prompt: System instructions.
        num_ctx: Context window size.
        host: Ollama server address.
        think: Enable/disable reasoning, or specify a supported level.
        format: Optional JSON mode or JSON schema.
        **kwargs: Additional Ollama inference options.

    Returns:
        Complete Ollama ChatResponse.
    """

    client = Client(host=host)

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    user_content = user_query

    if context is not None:
        user_content = f"""
<ontology>
{context}
</ontology>

<question>
{user_query}
</question>
"""

    messages.append({
        "role": "user",
        "content": user_content
    })

    options = {
        **kwargs,
        "num_ctx": num_ctx
    }

    response = client.chat(
        model=model,
        messages=messages,
        options=options,
        think=think,
        format=format,
        stream=False
    )

    return response

In [8]:
from typing import Literal
from pydantic import BaseModel, Field
import rdflib

MODEL = "gemma4:e4b"
ONTO_PATH = "../univ-bench-inst_no_individuals.txt"

# Testo dato come contesto al modello e grafo rdflib usato dal checker: stesso file.
with open(ONTO_PATH, encoding="utf-8") as f:
    lubm_onto_full = f.read()

onto_graph = rdflib.Graph()
onto_graph.parse(data=lubm_onto_full, format="turtle")
print(f"O: {len(onto_graph)} triples")

system_prompt = """
You are an expert in OWL ontology reasoning.

Answer questions using only the provided ontology.
Apply OWL semantics. Do not rely on external knowledge.

Return a JSON object with:
- "answer": "yes" or "no".
- "justification": a list of OWL axioms in Turtle that together entail the queried statement.

Rules for "justification":
* One axiom per list element; each element is a single Turtle statement ending with ".".
* Copy axioms exactly as they appear in the ontology; do not invent or derive axioms.
* Use the ":" prefix for ontology entities, plus rdf:, rdfs:, owl:. Do not write @prefix lines.
* Do not include class declarations (rdf:type owl:Class) or annotations (rdfs:label, rdfs:comment).
* Do not include the queried statement itself unless it is explicitly asserted in the ontology.
* If the answer is "no", justification must be an empty list [].
"""


class OntologyAnswer(BaseModel):
    answer: Literal["yes", "no"]
    justification: list[str] = Field(
        description="One OWL axiom per element, in Turtle. "
                    "Empty list if answer is 'no'."
    )
    # Nessun validatore yes <-> lista non vuota: una violazione è un dato dell'esperimento
    # e viene registrata dal checker (format_consistent) invece di far fallire il parsing.

O: 305 triples


In [9]:
# Query di sussunzione: "Is every SUB a SUP?"
# Altri esempi: ("UndergraduateStudent", "Student"), ("Publication", "Person")
# SUB, SUP = "AssistantProfessor", "Person"
# SUB, SUP = "Publication", "Person"
SUB, SUP = "UndergraduateStudent", "Person"
user_query = f"Is every {SUB} a {SUP}?"

response = generate_ollama_ans(
    model=MODEL,
    user_query=user_query,
    context=lubm_onto_full,
    system_prompt=system_prompt,
    num_ctx=32768,
    think=True,
    format=OntologyAnswer.model_json_schema(),  # JSON schema invece di "json" libero
    temperature=0.6,
    top_p=0.95,
    num_predict=-1,
    seed=42
)

result = OntologyAnswer.model_validate_json(response.message.content)
thinking = response.message.thinking

KeyboardInterrupt: 

In [15]:
result

OntologyAnswer(answer='yes', justification=[':UndergraduateStudent rdf:type owl:Class ; rdfs:subClassOf :Student.', ':Student rdf:type owl:Class ; owl:equivalentClass [ owl:intersectionOf ( :Person [ rdf:type owl:Restriction ; owl:onProperty :takesCourse ; owl:someValuesFrom :Course ] ) ; rdf:type owl:Class ] .'])

In [16]:
thinking

'The user wants to know if the class `UndergraduateStudent` is a subclass of `Person`.\nI need to check the class definitions in the provided ontology.\n\nLooking for `UndergraduateStudent`:\n`:UndergraduateStudent rdf:type owl:Class ; rdfs:subClassOf :Student ; rdfs:label "undergraduate student"`\n\nLooking for `Student`:\n`:Student rdf:type owl:Class ; owl:equivalentClass [ owl:intersectionOf ( :Person [ rdf:type owl:Restriction ; owl:onProperty :takesCourse ; owl:someValuesFrom :Course ] ) ; rdf:type owl:Class ] ; rdfs:label "student"`\n\nFrom the `Student` definition, it is equivalent to an intersection involving `:Person`.\nFrom the `UndergraduateStudent` definition, it is a subclass of `:Student`.\n\nSince `:UndergraduateStudent` subclass `:Student` and `:Student` is equivalent to something involving `:Person`, I need to check if `:UndergraduateStudent` must be a subclass of `:Person`.\n\nIf A subClassOf B, and B is equivalent to C, then A must be in the scope of C.\n\nThe defini

## Checker

In [11]:
import io
import shutil

import owlready2
import rdflib
from rdflib import RDF, RDFS, OWL, BNode, URIRef
from rdflib.collection import Collection

NS = "http://swat.cse.lehigh.edu/onto/univ-bench.owl#"
ONTO_IRI = "http://swat.cse.lehigh.edu/onto/univ-bench.owl"

# Intestazione aggiunta dal checker: il modello non scrive le righe @prefix.
TTL_HEADER = f"""@prefix : <{NS}> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
"""

# Triple ignorate nel confronto con O: dichiarazioni e annotazioni non sono assiomi logici.
DECLARATION_TYPES = {OWL.Class, OWL.ObjectProperty, OWL.DatatypeProperty,
                     OWL.AnnotationProperty, OWL.Ontology, OWL.Restriction}
ANNOTATION_PREDICATES = {RDFS.label, RDFS.comment, OWL.versionInfo}
SYMMETRIC_PREDICATES = {OWL.equivalentClass, OWL.equivalentProperty, OWL.inverseOf}
BUILTIN_NAMESPACES = (str(RDF), str(RDFS), str(OWL), "http://www.w3.org/2001/XMLSchema#")

java = shutil.which("java")
if not java:
    raise RuntimeError("Java required by HermiT; put the JDK on PATH.")
owlready2.JAVA_EXE = java


def _ignored(t):
    s, p, o = t
    return p in ANNOTATION_PREDICATES or (p == RDF.type and o in DECLARATION_TYPES)


def parse_axiom(axiom_ttl):
    """Parsa un singolo assioma Turtle prodotto dal modello. Ritorna (graph, errore)."""
    g = rdflib.Graph()
    try:
        g.parse(data=TTL_HEADER + axiom_ttl, format="turtle")
        return g, None
    except Exception as e:
        return None, f"{type(e).__name__}: {e}"


SET_PREDICATES = {OWL.intersectionOf, OWL.unionOf, OWL.oneOf}


def _canon(g, n):
    """Forma canonica di un nodo: i blank node (restrizioni, liste RDF) vengono confrontati
    per struttura; i membri di intersectionOf/unionOf/oneOf come insieme (ordine irrilevante)."""
    if not isinstance(n, BNode):
        return n.n3()
    if (n, RDF.first, None) in g:
        return ("list", tuple(_canon(g, i) for i in Collection(g, n)))
    props = []
    for p, o in g.predicate_objects(n):
        if _ignored((n, p, o)):
            continue
        c = _canon(g, o)
        if p in SET_PREDICATES and isinstance(c, tuple) and c[0] == "list":
            c = ("set", tuple(sorted(c[1], key=repr)))
        props.append((str(p), c))
    return ("node", tuple(sorted(props, key=repr)))


def axiom_in_ontology(ax_graph, onto_graph):
    """True se ogni tripla di primo livello dell'assioma (con eventuali blank node) è presente in O."""
    top = [t for t in ax_graph if not isinstance(t[0], BNode) and not _ignored(t)]
    if not top:
        return False, "no logical triple (only declarations/annotations or anonymous subject)"
    for s, p, o in top:
        if not isinstance(o, BNode):
            if (s, p, o) in onto_graph:
                continue
            if p in SYMMETRIC_PREDICATES and (o, p, s) in onto_graph:
                continue
            return False, f"triple not in O: {s.n3(onto_graph.namespace_manager)} {p.n3(onto_graph.namespace_manager)} {o.n3(onto_graph.namespace_manager)}"
        target = _canon(ax_graph, o)
        candidates = [o2 for o2 in onto_graph.objects(s, p) if isinstance(o2, BNode)]
        if not any(_canon(onto_graph, o2) == target for o2 in candidates):
            return False, f"no matching anonymous expression in O for {s.n3(onto_graph.namespace_manager)} {p.n3(onto_graph.namespace_manager)} [...]"
    return True, None


def signature(g):
    return {n for t in g for n in t
            if isinstance(n, URIRef) and not str(n).startswith(BUILTIN_NAMESPACES)}


def build_justification_graph(ax_graphs, onto_graph, query_classes):
    """Unisce gli assiomi e aggiunge le dichiarazioni delle entità prese da O."""
    j = rdflib.Graph()
    j.add((URIRef(ONTO_IRI), RDF.type, OWL.Ontology))
    for g in ax_graphs:
        j += g
    for e in signature(j) | set(query_classes):
        for typ in onto_graph.objects(e, RDF.type):
            j.add((e, RDF.type, typ))
    return j


def hermit_entails_subclass(graph, sub, sup):
    """True se graph ⊨ sub ⊑ sup secondo HermiT (via owlready2)."""
    world = owlready2.World()
    try:
        world.get_ontology(ONTO_IRI).load(
            fileobj=io.BytesIO(graph.serialize(format="xml", encoding="utf-8")))
        owlready2.sync_reasoner(world, infer_property_values=False, debug=0)
        c_sub, c_sup = world[str(sub)], world[str(sup)]
        if c_sub is None or c_sup is None:
            return False
        return c_sub == c_sup or c_sup in c_sub.ancestors()
    finally:
        world.close()


def check_answer(result, sub, sup, onto_graph, ground_truth=None):
    """Verifica la risposta strutturata del modello per la query sub ⊑ sup."""
    sub, sup = URIRef(NS + sub), URIRef(NS + sup)
    onto_sig = signature(onto_graph)
    query_triple = (sub, RDFS.subClassOf, sup)

    if ground_truth is None:
        ground_truth = hermit_entails_subclass(onto_graph, sub, sup)
    gt_answer = "yes" if ground_truth else "no"

    axioms = []
    for ax in result.justification:
        g, err = parse_axiom(ax)
        rec = {"axiom": ax, "parsed": g is not None, "parse_error": err}
        if g is not None:
            rec["unknown_entities"] = sorted(str(e).replace(NS, ":") for e in signature(g) - onto_sig)
            rec["in_O"], rec["not_in_O_reason"] = axiom_in_ontology(g, onto_graph)
            rec["is_query_axiom"] = query_triple in g
        axioms.append((rec, g))

    report = {
        "query": f"{sub.replace(NS, ':')} rdfs:subClassOf {sup.replace(NS, ':')}",
        "llm_answer": result.answer,
        "ground_truth": gt_answer,
        "answer_correct": result.answer == gt_answer,
        # yes <-> justification non vuota, no <-> justification vuota
        "format_consistent": (result.answer == "yes") == bool(result.justification),
        "n_axioms": len(axioms),
        "all_parsed": all(r["parsed"] for r, _ in axioms),
        "all_in_O": all(r.get("in_O", False) for r, _ in axioms),
        "justification_entails_query": None,
        "certificate_valid": None,
        "axioms": [r for r, _ in axioms],
    }

    if result.answer == "yes" and not axioms:
        report["certificate_valid"] = False  # "yes" senza certificato
    elif result.answer == "yes":
        parsed = [g for r, g in axioms if g is not None]
        j_graph = build_justification_graph(parsed, onto_graph, [sub, sup])
        report["justification_entails_query"] = hermit_entails_subclass(j_graph, sub, sup)
        # Certificato valido: assiomi tutti presenti in O e sufficienti per la query.
        report["certificate_valid"] = (report["all_parsed"] and report["all_in_O"]
                                       and report["justification_entails_query"])
    return report

In [12]:
import json

report = check_answer(result, SUB, SUP, onto_graph)

summary_keys = ["query", "llm_answer", "ground_truth", "answer_correct", "format_consistent",
                "n_axioms", "all_parsed", "all_in_O", "justification_entails_query", "certificate_valid"]
for k in summary_keys:
    print(f"{k:28} {report[k]}")

print("\nAxioms:")
for a in report["axioms"]:
    ok = a["parsed"] and a.get("in_O") and not a.get("unknown_entities")
    print(("  ✓ " if ok else "  ✗ ") + a["axiom"])
    for k in ["parse_error", "unknown_entities", "not_in_O_reason"]:
        if a.get(k):
            print(f"      {k}: {a[k]}")
    if a.get("is_query_axiom"):
        print("      warning: contains the queried axiom")

NameError: name 'result' is not defined

In [ ]:
## come funziona check_answer
# check_answer(result, sub, sup, onto_graph, ground_truth=None)
# ├─ signature(onto_graph)                      # vocabolario di O
# ├─ hermit_entails_subclass(onto_graph, ...)   # ground truth (solo se ground_truth è None)
# ├─ per ogni stringa in result.justification:
# │   ├─ parse_axiom(ax)                        # stringa Turtle → rdflib.Graph
# │   ├─ signature(g) − signature(O)            # entità sconosciute
# │   └─ axiom_in_ontology(g, onto_graph)       # presenza in O
# │        ├─ _ignored(t)
# │        └─ _canon(g, bnode)                  # confronto strutturale dei blank node
# └─ se answer == "yes" e la lista non è vuota:
#     ├─ build_justification_graph(...)         # ontologia contenente solo J
#     └─ hermit_entails_subclass(J, ...)        # J ⊨ sub ⊑ sup ?

## Esperimento su tutto il dataset

Si usano gli elementi di `owlapy_generated_qa.json` che hanno `sub` e `sup` (query di sussunzione).
Le query di istanza (`individual`/`class`) e quelle transitive sono escluse: il contesto dato al modello
non contiene individui e il checker verifica solo `sub ⊑ sup`.

I risultati vengono scritti in `{MODEL}_answers_w_checker.jsonl`, **una riga per query, appena calcolata**.
Se l'esecuzione si interrompe, rilanciando la cella le query già presenti nel file (stesso `idx`) vengono saltate.

In [23]:
import json
from collections import Counter

DATASET_PATH = "../owlapy_generated_qa.json"

with open(DATASET_PATH, encoding="utf-8") as f:
    generated_queries = json.load(f)

# idx = posizione nel file originale: identifica la query anche per la ripresa
dataset = [{"idx": i, **q} for i, q in enumerate(generated_queries) if "sub" in q and "sup" in q]

print(f"{len(generated_queries)} query nel file, {len(dataset)} di sussunzione")
for (rt, ans), n in sorted(Counter((q["reasoning_type"], q["answer"]) for q in dataset).items()):
    print(f"  {rt:22} answer={ans!s:5}  {n}")

187 query nel file, 152 di sussunzione
  subClassOf_direct      answer=True   38
  subClassOf_inferred    answer=True   38
  subClassOf_negative    answer=False  76


In [24]:
# Ground truth: O viene classificata con HermiT una sola volta, poi si leggono gli antenati.
def hermit_superclass_closure(graph):
    world = owlready2.World()
    try:
        world.get_ontology(ONTO_IRI).load(
            fileobj=io.BytesIO(graph.serialize(format="xml", encoding="utf-8")))
        owlready2.sync_reasoner(world, infer_property_values=False, debug=0)
        return {c.iri: {a.iri for a in c.ancestors() if hasattr(a, "iri")} for c in world.classes()}
    finally:
        world.close()

O_CLOSURE = hermit_superclass_closure(onto_graph)

def ground_truth_of(sub, sup):
    return NS + sup in O_CLOSURE.get(NS + sub, set())

# Controllo di coerenza: etichette del dataset vs HermiT
mismatch = [q for q in dataset if ground_truth_of(q["sub"], q["sup"]) != q["answer"]]
print(f"dataset vs HermiT: {len(dataset) - len(mismatch)}/{len(dataset)} concordi")
for q in mismatch:
    print("  mismatch:", q["idx"], q["question"], "dataset =", q["answer"])

dataset vs HermiT: 152/152 concordi


In [30]:
import os
import time
from pydantic import ValidationError

# Nome file: i caratteri ":" e "/" del nome del modello (es. "gemma4:e4b") vengono sostituiti
OUT_PATH = f"{MODEL.replace(':', '_').replace('/', '_')}_answers_w_checker.jsonl"

GEN_OPTIONS = dict(
    num_ctx=32768,
    think=True,
    temperature=0.6,
    top_p=0.95,
    num_predict=-1,
    seed=42,
    repeat_penalty= 1.1
)


def load_done(path):
    """idx delle query già registrate nel file di output."""
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    done.add(json.loads(line)["idx"])
    return done


def run_one(q):
    """Interroga il modello su una query del dataset, esegue il checker e ritorna il record."""
    t0 = time.time()
    response = generate_ollama_ans(
        model=MODEL,
        user_query=q["question"],
        context=lubm_onto_full,
        system_prompt=system_prompt,
        format=OntologyAnswer.model_json_schema(),
        **GEN_OPTIONS,
    )
    latency = time.time() - t0

    record = {
        "idx": q["idx"],
        "model": MODEL,
        "question": q["question"],
        "reasoning_type": q["reasoning_type"],
        "sub": q["sub"],
        "sup": q["sup"],
        "dataset_answer": "yes" if q["answer"] else "no",
        "raw_output": response.message.content,
        "llm_output_error": None,   # JSON non valido / non conforme allo schema
        "checker_error": None,      # eccezione nel checker (es. ontologia inconsistente)
        "llm_justification": None,
        "report": None,             # tutto ciò che ritorna check_answer()
        "thinking": getattr(response.message, "thinking", None),
        "latency_s": round(latency, 2),
        "prompt_eval_count": getattr(response, "prompt_eval_count", None),
        "eval_count": getattr(response, "eval_count", None),
        "gen_options": GEN_OPTIONS,
    }

    try:
        result = OntologyAnswer.model_validate_json(response.message.content)
    except ValidationError as e:
        record["llm_output_error"] = str(e)
        return record

    record["llm_justification"] = result.justification
    try:
        record["report"] = check_answer(result, q["sub"], q["sup"], onto_graph,
                                        ground_truth=ground_truth_of(q["sub"], q["sup"]))
    except Exception as e:
        record["checker_error"] = f"{type(e).__name__}: {e}"
    return record


def run_dataset(queries, out_path=OUT_PATH):
    done = load_done(out_path)
    todo = [q for q in queries if q["idx"] not in done]
    print(f"{len(done)} già nel file, {len(todo)} da eseguire → {out_path}")

    with open(out_path, "a", encoding="utf-8") as out:
        for n, q in enumerate(todo, 1):
            try:
                record = run_one(q)
            except Exception as e:
                # Errore lato Ollama (server spento, modello assente...): la query non viene
                # scritta, così al prossimo lancio verrà ritentata.
                print(f"[{n}/{len(todo)}] idx={q['idx']} ERRORE Ollama: {type(e).__name__}: {e}")
                continue
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()

            rep = record["report"] or {}
            status = (record["llm_output_error"] and "output non valido") or \
                     (record["checker_error"] and "errore checker") or \
                     f"answer={rep.get('llm_answer')} gt={rep.get('ground_truth')} cert={rep.get('certificate_valid')}"
            print(f"[{n}/{len(todo)}] idx={q['idx']} {q['sub']} ⊑ {q['sup']}: {status}  ({record['latency_s']}s)")

In [31]:
run_dataset(dataset)

151 già nel file, 1 da eseguire → gemma4_e4b_answers_w_checker.jsonl
[1/1] idx=40 VisitingProfessor ⊑ Person: answer=yes gt=yes cert=False  (20.91s)


In [17]:
# Riepilogo dal file jsonl (si può eseguire anche a run in corso)
def summarize(path=OUT_PATH):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]
    print(f"{len(rows)} record in {path}")
    print(f"  output non valido: {sum(r['llm_output_error'] is not None for r in rows)}"
          f" | errori checker: {sum(r['checker_error'] is not None for r in rows)}\n")

    ok = [r for r in rows if r["report"]]
    groups = {}
    for r in ok:
        groups.setdefault(r["reasoning_type"], []).append(r["report"])
    groups["TOTALE"] = [r["report"] for r in ok]

    def pct(xs):
        return f"{100 * sum(xs) / len(xs):5.1f}%" if xs else "   - "

    print(f"{'reasoning_type':22} {'n':>4} {'accuracy':>9} {'formato':>8} {'cert. valid (su yes)':>21} {'giusta ma cert. non valido':>27}")
    for name, reps in groups.items():
        yes = [x for x in reps if x["llm_answer"] == "yes"]
        right_bad = [x for x in reps if x["answer_correct"] and x["llm_answer"] == "yes" and not x["certificate_valid"]]
        print(f"{name:22} {len(reps):4} {pct([x['answer_correct'] for x in reps]):>9} "
              f"{pct([x['format_consistent'] for x in reps]):>8} "
              f"{pct([bool(x['certificate_valid']) for x in yes]):>21} {len(right_bad):>27}")

summarize()

151 record in gemma4_e4b_answers_w_checker.jsonl
  output non valido: 0 | errori checker: 0

reasoning_type            n  accuracy  formato  cert. valid (su yes)  giusta ma cert. non valido
subClassOf_direct        38     92.1%   100.0%                 88.6%                           4
subClassOf_inferred      37     78.4%   100.0%                 48.3%                          15
subClassOf_negative      76    100.0%   100.0%                    -                            0
TOTALE                  151     92.7%   100.0%                 70.3%                          19


In [26]:
## analisi del thinking tramite faithfulness e altre metriche 
## analisi lunghezza: se sono troppo corte non conviene usare judge, si vede solo se citano gli assiomi necessari. 
import json 
import matplotlib.pyplot as plt
with open("./gemma4_e4b_answers_w_checker.jsonl") as f:
    gemma4_ans = [json.loads(i) for i in f]
# gemma4_ans
gemma4_ans_thinkings = [i['thinking'] for i in gemma4_ans]
print("thinking: \n", gemma4_ans_thinkings[-4], "justification: \n", gemma4_ans[-4]['llm_justification'])
# gemma4_ans_thinkings_len = [len(i) for i in gemma4_ans_thinkings]
# gemma4_ans_thinkings_len


thinking: 
 1.  **Analyze the Request:** The user wants to know if `VisitingProfessor` is a subclass of `ClericalStaff` based on the provided OWL ontology.

2.  **Identify Relevant Classes:**
    *   `VisitingProfessor`: Defined as `rdfs:subClassOf :Professor`.
    *   `ClericalStaff`: Defined as `rdfs:subClassOf :AdministrativeStaff`.
    *   `Professor`: Defined as a subclass of `Faculty`.
    *   `AdministrativeStaff`: Defined as a subclass of `Employee`.

3.  **Check Subclass Relationships (Direct and Transitive):**
    *   Is `VisitingProfessor` (subClassOf `Professor`) related to `ClericalStaff` (subClassOf `AdministrativeStaff`)?
    *   `VisitingProfessor` is a type of `Professor`.
    *   `ClericalStaff` is a type of `AdministrativeStaff`.
    *   The ontology does not assert any relationship (subClassOf, equivalentClass, etc.) between `Professor` and `AdministrativeStaff`, or between `VisitingProfessor` and `ClericalStaff`.

4.  **Formulate Conclusion:** Since there is no axi

### esperimento robusto 

In [ ]:
## testa temperature 0  e vari seed 


## esperimento gemma4 senza chiedere proof


In [41]:
from ollama import Client
from ollama import ChatResponse
from typing import Optional, Any
import rdflib
import os
import time
from pydantic import ValidationError

system_prompt_exp_no_checker = """
You are an expert in OWL ontology reasoning.

Answer questions using only the provided ontology.
Apply OWL semantics. Do not rely on external knowledge.

Reply "yes" to the user's question if the query is entailed in the ontology. 
Otherwise reply "no". 

Only return "yes" or "no" and do not add any additional text. 
"""

MODEL = "gemma4:e4b"
ONTO_PATH = "../univ-bench-inst_no_individuals.txt"

# Testo dato come contesto al modello e grafo rdflib usato dal checker: stesso file.
with open(ONTO_PATH, encoding="utf-8") as f:
    lubm_onto_full = f.read()

onto_graph = rdflib.Graph()
onto_graph.parse(data=lubm_onto_full, format="turtle")
print(f"O: {len(onto_graph)} triples")

## funzioni modificate per non usare checker

def generate_ollama_ans(
    model: str,
    user_query: str,
    context: Optional[str] = None,
    system_prompt: Optional[str] = None,
    num_ctx: int = 8192,
    host: str = "http://localhost:11434",
    think: Optional[bool | str] = None,
    format: Optional[str | dict] = None,
    **kwargs: Any
) -> ChatResponse:
    """
    Generate a response using a local Ollama model.

    Args:
        model: Name of the Ollama model.
        user_query: Question to answer.
        context: Optional ontology or other reasoning context.
        system_prompt: System instructions.
        num_ctx: Context window size.
        host: Ollama server address.
        think: Enable/disable reasoning, or specify a supported level.
        format: Optional JSON mode or JSON schema.
        **kwargs: Additional Ollama inference options.

    Returns:
        Complete Ollama ChatResponse.
    """

    client = Client(host=host)

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    user_content = user_query

    if context is not None:
        user_content = f"""
<ontology>
{context}
</ontology>

<question>
{user_query}
</question>
"""

    messages.append({
        "role": "user",
        "content": user_content
    })

    options = {
        **kwargs,
        "num_ctx": num_ctx
    }

    response = client.chat(
        model=model,
        messages=messages,
        options=options,
        think=think,
        format=format,
        stream=False
    )

    return response

OUT_PATH = f"{MODEL.replace(':', '_').replace('/', '_')}_answers_NO_checker.jsonl"

GEN_OPTIONS = dict(
    num_ctx=32768,
    think=True,
    temperature=0.6,
    top_p=0.95,
    num_predict=-1,
    seed=42,
)


def load_done(path):
    """idx delle query già registrate nel file di output."""
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    done.add(json.loads(line)["idx"])
    return done


def run_one_no_checker(q):
    """Interroga il modello su una query del dataset, esegue il checker e ritorna il record."""
    t0 = time.time()
    response = generate_ollama_ans(
        model=MODEL,
        user_query=q["question"],
        context=lubm_onto_full,
        system_prompt=system_prompt_exp_no_checker,
        # format=OntologyAnswer.model_json_schema(),
        **GEN_OPTIONS,
    )
    latency = time.time() - t0

    record = {
        "idx": q["idx"],
        "model": MODEL,
        "question": q["question"],
        "reasoning_type": q["reasoning_type"],
        "sub": q["sub"],
        "sup": q["sup"],
        "dataset_answer": "yes" if q["answer"] else "no",
        "raw_output": response.message.content,
        "thinking": getattr(response.message, "thinking", None),
        "latency_s": round(latency, 2),
        "prompt_eval_count": getattr(response, "prompt_eval_count", None), ## the number of input tokens contained in your prompt
        "eval_count": getattr(response, "eval_count", None), ## number of output tokens generated by the model in a response
        "gen_options": GEN_OPTIONS,
    }
    return record


def run_dataset(queries, out_path=OUT_PATH):
    done = load_done(out_path)
    todo = [q for q in queries if q["idx"] not in done]
    print(f"{len(done)} già nel file, {len(todo)} da eseguire → {out_path}")

    with open(out_path, "a", encoding="utf-8") as out:
        for n, q in enumerate(todo, 1):
            try:
                record = run_one_no_checker(q)
            except Exception as e:
                # Errore lato Ollama (server spento, modello assente...): la query non viene
                # scritta, così al prossimo lancio verrà ritentata.
                print(f"[{n}/{len(todo)}] idx={q['idx']} ERRORE Ollama: {type(e).__name__}: {e}")
                continue
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()

            # rep = record["report"] or {}
            # status = (record["llm_output_error"] and "output non valido") or \
            #          (record["checker_error"] and "errore checker") or \
            #          f"answer={rep.get('llm_answer')} gt={rep.get('ground_truth')} cert={rep.get('certificate_valid')}"
            print(f"[{n}/{len(todo)}] idx={q['idx']} {q['sub']} ⊑ {q['sup']}:  ({record['latency_s']}s)")



O: 305 triples


In [44]:
import json
from collections import Counter

DATASET_PATH = "../owlapy_generated_qa.json"

with open(DATASET_PATH, encoding="utf-8") as f:
    generated_queries = json.load(f)

# idx = posizione nel file originale: identifica la query anche per la ripresa
dataset = [{"idx": i, **q} for i, q in enumerate(generated_queries) if "sub" in q and "sup" in q]

print(f"{len(generated_queries)} query nel file, {len(dataset)} di sussunzione")
for (rt, ans), n in sorted(Counter((q["reasoning_type"], q["answer"]) for q in dataset).items()):
    print(f"  {rt:22} answer={ans!s:5}  {n}")


187 query nel file, 152 di sussunzione
  subClassOf_direct      answer=True   38
  subClassOf_inferred    answer=True   38
  subClassOf_negative    answer=False  76


In [45]:
    
run_dataset(dataset)

0 già nel file, 152 da eseguire → gemma4_e4b_answers_NO_checker.jsonl
[1/152] idx=0 VisitingProfessor ⊑ Professor:  (0.26s)
[2/152] idx=1 UnofficialPublication ⊑ Publication:  (10.55s)
[3/152] idx=2 University ⊑ Organization:  (10.49s)
[4/152] idx=3 UndergraduateStudent ⊑ Student:  (0.2s)
[5/152] idx=4 TechnicalReport ⊑ Article:  (12.04s)
[6/152] idx=5 TeachingAssistant ⊑ Person:  (24.32s)
[7/152] idx=6 SystemsStaff ⊑ AdministrativeStaff:  (0.21s)
[8/152] idx=7 Student ⊑ Person:  (16.43s)
[9/152] idx=8 Specification ⊑ Publication:  (9.32s)
[10/152] idx=9 Software ⊑ Publication:  (8.75s)
[11/152] idx=10 ResearchGroup ⊑ Organization:  (7.14s)
[12/152] idx=11 ResearchAssistant ⊑ Employee:  (28.11s)
[13/152] idx=12 Research ⊑ Work:  (9.94s)
[14/152] idx=13 Program ⊑ Organization:  (9.9s)
[15/152] idx=14 Professor ⊑ Faculty:  (12.21s)
[16/152] idx=15 PostDoc ⊑ Faculty:  (12.15s)
[17/152] idx=16 Manual ⊑ Publication:  (8.55s)
[18/152] idx=17 Lecturer ⊑ Faculty:  (8.52s)
[19/152] idx=18 Journ

In [ ]:
import json
OUT_PATH = "gemma4_e4b_answers_NO_checker.jsonl"

with open(OUT_PATH, encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]
gemma4_no_check_discrete_ans = [i['raw_output'] for i in rows]
gold_
gemma4_no_check_discrete_ans

['yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'yes',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',
 'no',

In [5]:
# Riepilogo dal file jsonl (si può eseguire anche a run in corso)
# Senza checker il record non ha "report": la risposta è il testo grezzo in raw_output
# e la ground truth è dataset_answer.
def summarize(path=OUT_PATH):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]
    print(f"{len(rows)} record in {path}")

    def parse_answer(raw):
        """'yes'/'no' normalizzato, None se l'output non è una risposta valida."""
        a = (raw or "").strip().strip('."\'`* ').lower()
        return a if a in ("yes", "no") else None

    reps = [{"llm_answer": parse_answer(r["raw_output"]),
             "ground_truth": r["dataset_answer"],
             "reasoning_type": r["reasoning_type"]} for r in rows]
    for x in reps:
        x["answer_correct"] = x["llm_answer"] == x["ground_truth"]
    print(f"  output non valido: {sum(x['llm_answer'] is None for x in reps)}\n")

    groups = {}
    for x in reps:
        groups.setdefault(x["reasoning_type"], []).append(x)
    groups["TOTALE"] = reps

    def pct(xs):
        return f"{100 * sum(xs) / len(xs):5.1f}%" if xs else "   - "

    print(f"{'reasoning_type':22} {'n':>4} {'accuracy':>9} {'formato':>8} {'acc. su gt=yes':>15} {'acc. su gt=no':>14}")
    for name, g in groups.items():
        gt_yes = [x for x in g if x["ground_truth"] == "yes"]
        gt_no = [x for x in g if x["ground_truth"] == "no"]
        print(f"{name:22} {len(g):4} {pct([x['answer_correct'] for x in g]):>9} "
              f"{pct([x['llm_answer'] is not None for x in g]):>8} "
              f"{pct([x['answer_correct'] for x in gt_yes]):>15} {pct([x['answer_correct'] for x in gt_no]):>14}")

summarize()

152 record in gemma4_e4b_answers_NO_checker.jsonl
  output non valido: 0

reasoning_type            n  accuracy  formato  acc. su gt=yes  acc. su gt=no
subClassOf_direct        38     97.4%   100.0%           97.4%             - 
subClassOf_inferred      38     86.8%   100.0%           86.8%             - 
subClassOf_negative      76    100.0%   100.0%              -          100.0%
TOTALE                  152     96.1%   100.0%           92.1%         100.0%


In [18]:
# Confronto: esperimento con checker (risposta + justification) vs senza checker (solo yes/no).
# Si confrontano solo le query presenti in entrambi i file (stesso idx).
W_CHECKER_PATH = "gemma4_e4b_answers_w_checker.jsonl"
NO_CHECKER_PATH = "gemma4_e4b_answers_NO_checker.jsonl"


def compare(w_path=W_CHECKER_PATH, no_path=NO_CHECKER_PATH):
    def load(path):
        with open(path, encoding="utf-8") as f:
            return {r["idx"]: r for r in (json.loads(l) for l in f if l.strip())}

    w, no = load(w_path), load(no_path)
    common = sorted(w.keys() & no.keys())
    print(f"con checker: {len(w)} | senza checker: {len(no)} | in comune: {len(common)}")
    for name, only in [("solo con checker", w.keys() - no.keys()), ("solo senza checker", no.keys() - w.keys())]:
        if only:
            print(f"  {name}: idx {sorted(only)}")

    rows = []
    for i in common:
        rw, rn = w[i], no[i]
        rep = rw["report"] or {}
        ans_no = (rn["raw_output"] or "").strip().strip('."\'`* ').lower()
        rows.append({
            "idx": i,
            "reasoning_type": rw["reasoning_type"],
            "question": rw["question"],
            "gt": rw["dataset_answer"],
            "ans_w": rep.get("llm_answer"),          # None se output non valido / errore checker
            "ans_no": ans_no if ans_no in ("yes", "no") else None,
            "cert_valid": bool(rep.get("certificate_valid")),
            "lat_w": rw["latency_s"], "lat_no": rn["latency_s"],
            "tok_w": rw["eval_count"] or 0, "tok_no": rn["eval_count"] or 0,
        })
    for r in rows:
        r["ok_w"] = r["ans_w"] == r["gt"]
        r["ok_no"] = r["ans_no"] == r["gt"]
        # risposta corretta e, se "yes", con certificato valido
        r["ok_w_cert"] = r["ok_w"] and (r["ans_w"] == "no" or r["cert_valid"])

    groups = {}
    for r in rows:
        groups.setdefault(r["reasoning_type"], []).append(r)
    groups["TOTALE"] = rows

    def pct(xs):
        return f"{100 * sum(xs) / len(xs):5.1f}%" if xs else "   - "

    def mean(xs):
        return f"{sum(xs) / len(xs):.1f}" if xs else "-"

    print(f"\n{'reasoning_type':22} {'n':>4} {'acc. NO chk':>11} {'acc. W chk':>11} {'acc. W chk+cert':>16}"
          f" {'entrambe ok':>12} {'solo NO':>8} {'solo W':>7} {'entrambe ko':>12}")
    for name, g in groups.items():
        print(f"{name:22} {len(g):4} {pct([r['ok_no'] for r in g]):>11} {pct([r['ok_w'] for r in g]):>11} "
              f"{pct([r['ok_w_cert'] for r in g]):>16} "
              f"{sum(r['ok_w'] and r['ok_no'] for r in g):>12} "
              f"{sum(r['ok_no'] and not r['ok_w'] for r in g):>8} "
              f"{sum(r['ok_w'] and not r['ok_no'] for r in g):>7} "
              f"{sum(not r['ok_w'] and not r['ok_no'] for r in g):>12}")

    print(f"\n{'reasoning_type':22} {'latenza NO (s)':>15} {'latenza W (s)':>14} {'token out NO':>13} {'token out W':>12}")
    for name, g in groups.items():
        print(f"{name:22} {mean([r['lat_no'] for r in g]):>15} {mean([r['lat_w'] for r in g]):>14} "
              f"{mean([r['tok_no'] for r in g]):>13} {mean([r['tok_w'] for r in g]):>12}")

    diff = [r for r in rows if r["ok_w"] != r["ok_no"]]
    print(f"\nQuery con esito diverso ({len(diff)}):")
    for r in diff:
        print(f"  idx={r['idx']:<4} {r['reasoning_type']:20} gt={r['gt']:3} NO chk={r['ans_no']!s:4} "
              f"W chk={r['ans_w']!s:4} cert={r['cert_valid']!s:5}  {r['question']}")
    return rows


comparison_rows = compare()

con checker: 151 | senza checker: 152 | in comune: 151
  solo senza checker: idx [40]

reasoning_type            n acc. NO chk  acc. W chk  acc. W chk+cert  entrambe ok  solo NO  solo W  entrambe ko
subClassOf_direct        38       97.4%       92.1%            81.6%           35        2       0            1
subClassOf_inferred      37       86.5%       78.4%            37.8%           27        5       2            3
subClassOf_negative      76      100.0%      100.0%           100.0%           76        0       0            0
TOTALE                  151       96.0%       92.7%            80.1%          138        7       2            4

reasoning_type          latenza NO (s)  latenza W (s)  token out NO  token out W
subClassOf_direct                  9.1            9.9         247.2        250.7
subClassOf_inferred               21.6           25.4         595.9        692.0
subClassOf_negative               12.8           13.9         357.6        387.2
TOTALE                      

## Esperimento qwen3.5:4b (con e senza checker)

Stessa logica, stessi dati e stesse opzioni di generazione degli esperimenti gemma4:

- **con checker**: `system_prompt` + JSON schema `OntologyAnswer`, risposta verificata con `check_answer`;
- **senza checker**: `system_prompt_exp_no_checker`, risposta libera `yes`/`no`;
- dataset: le query di sussunzione di `owlapy_generated_qa.json` (`dataset`), ground truth da `ground_truth_of`.

I risultati vanno in file separati, **senza toccare quelli di gemma4**:
`qwen3.5_4b_answers_w_checker.jsonl` e `qwen3.5_4b_answers_NO_checker.jsonl`
(scrittura in append, una riga per query; rilanciando la cella le query già presenti vengono saltate).

Le funzioni di questa sezione ricevono modello e file di output come parametri, quindi non dipendono
dai globali `MODEL` / `OUT_PATH` / `run_dataset` ridefiniti nelle sezioni precedenti.

**Prerequisiti** (eseguire prima): celle con `generate_ollama_ans`, `system_prompt` + `OntologyAnswer` + `onto_graph`,
checker (`check_answer`), `dataset`, ground truth (`O_CLOSURE` / `ground_truth_of`), `system_prompt_exp_no_checker`
(prima cella della sezione "esperimento gemma4 senza chiedere proof") e `compare`.

In [ ]:
import json
import os
import time
from pydantic import ValidationError

QWEN_MODEL = "qwen3.5:4b"
QWEN_TAG = QWEN_MODEL.replace(":", "_").replace("/", "_")
QWEN_W_CHECKER_PATH = f"{QWEN_TAG}_answers_w_checker.jsonl"
QWEN_NO_CHECKER_PATH = f"{QWEN_TAG}_answers_NO_checker.jsonl"

# Stesse opzioni di generazione dei run gemma4
QWEN_GEN_OPTIONS = dict(
    num_ctx=32768,
    think=True,
    temperature=0.6,
    top_p=0.95,
    num_predict=-1,
    seed=42,
)

# Protezione: i file qwen non devono coincidere con quelli di gemma4
for path in (QWEN_W_CHECKER_PATH, QWEN_NO_CHECKER_PATH):
    assert not path.startswith("gemma4"), path


def load_done_idx(path):
    """idx delle query già registrate nel file di output."""
    done = set()
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    done.add(json.loads(line)["idx"])
    return done


def run_one_w_checker_model(q, model, gen_options):
    """Come run_one: risposta strutturata (answer + justification) verificata con check_answer."""
    t0 = time.time()
    response = generate_ollama_ans(
        model=model,
        user_query=q["question"],
        context=lubm_onto_full,
        system_prompt=system_prompt,
        format=OntologyAnswer.model_json_schema(),
        **gen_options,
    )
    latency = time.time() - t0

    record = {
        "idx": q["idx"],
        "model": model,
        "question": q["question"],
        "reasoning_type": q["reasoning_type"],
        "sub": q["sub"],
        "sup": q["sup"],
        "dataset_answer": "yes" if q["answer"] else "no",
        "raw_output": response.message.content,
        "llm_output_error": None,   # JSON non valido / non conforme allo schema
        "checker_error": None,      # eccezione nel checker (es. ontologia inconsistente)
        "llm_justification": None,
        "report": None,             # tutto ciò che ritorna check_answer()
        "thinking": getattr(response.message, "thinking", None),
        "latency_s": round(latency, 2),
        "prompt_eval_count": getattr(response, "prompt_eval_count", None),
        "eval_count": getattr(response, "eval_count", None),
        "gen_options": gen_options,
    }

    try:
        result = OntologyAnswer.model_validate_json(response.message.content)
    except ValidationError as e:
        record["llm_output_error"] = str(e)
        return record

    record["llm_justification"] = result.justification
    try:
        record["report"] = check_answer(result, q["sub"], q["sup"], onto_graph,
                                        ground_truth=ground_truth_of(q["sub"], q["sup"]))
    except Exception as e:
        record["checker_error"] = f"{type(e).__name__}: {e}"
    return record


def run_one_no_checker_model(q, model, gen_options):
    """Come run_one_no_checker: risposta libera yes/no, nessuna justification."""
    t0 = time.time()
    response = generate_ollama_ans(
        model=model,
        user_query=q["question"],
        context=lubm_onto_full,
        system_prompt=system_prompt_exp_no_checker,
        **gen_options,
    )
    latency = time.time() - t0

    return {
        "idx": q["idx"],
        "model": model,
        "question": q["question"],
        "reasoning_type": q["reasoning_type"],
        "sub": q["sub"],
        "sup": q["sup"],
        "dataset_answer": "yes" if q["answer"] else "no",
        "raw_output": response.message.content,
        "thinking": getattr(response.message, "thinking", None),
        "latency_s": round(latency, 2),
        "prompt_eval_count": getattr(response, "prompt_eval_count", None),
        "eval_count": getattr(response, "eval_count", None),
        "gen_options": gen_options,
    }


def run_dataset_model(queries, run_one_fn, model, out_path, gen_options):
    """Esegue run_one_fn su ogni query non ancora presente in out_path (append, ripresa per idx)."""
    done = load_done_idx(out_path)
    todo = [q for q in queries if q["idx"] not in done]
    print(f"{model}: {len(done)} già nel file, {len(todo)} da eseguire → {out_path}")

    with open(out_path, "a", encoding="utf-8") as out:
        for n, q in enumerate(todo, 1):
            try:
                record = run_one_fn(q, model, gen_options)
            except Exception as e:
                # Errore lato Ollama: la query non viene scritta e sarà ritentata al prossimo lancio.
                print(f"[{n}/{len(todo)}] idx={q['idx']} ERRORE Ollama: {type(e).__name__}: {e}")
                continue
            out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()

            if "report" in record:
                rep = record["report"] or {}
                status = (record["llm_output_error"] and "output non valido") or \
                         (record["checker_error"] and "errore checker") or \
                         f"answer={rep.get('llm_answer')} gt={rep.get('ground_truth')} cert={rep.get('certificate_valid')}"
            else:
                status = f"answer={record['raw_output'].strip()[:20]!r} gt={record['dataset_answer']}"
            print(f"[{n}/{len(todo)}] idx={q['idx']} {q['sub']} ⊑ {q['sup']}: {status}  ({record['latency_s']}s)")

In [ ]:
# qwen3.5:4b con checker
run_dataset_model(dataset, run_one_w_checker_model, QWEN_MODEL, QWEN_W_CHECKER_PATH, QWEN_GEN_OPTIONS)

In [ ]:
# qwen3.5:4b senza checker
run_dataset_model(dataset, run_one_no_checker_model, QWEN_MODEL, QWEN_NO_CHECKER_PATH, QWEN_GEN_OPTIONS)

In [21]:
# Riepilogo dei due run qwen (stesse metriche dei riepiloghi gemma4).
# Funzioni con nome proprio: "summarize" è definita due volte sopra con formati diversi.
def pct(xs):
    return f"{100 * sum(xs) / len(xs):5.1f}%" if xs else "   - "


def summarize_w_checker(path):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]
    print(f"{len(rows)} record in {path}")
    print(f"  output non valido: {sum(r['llm_output_error'] is not None for r in rows)}"
          f" | errori checker: {sum(r['checker_error'] is not None for r in rows)}\n")

    ok = [r for r in rows if r["report"]]
    groups = {}
    for r in ok:
        groups.setdefault(r["reasoning_type"], []).append(r["report"])
    groups["TOTALE"] = [r["report"] for r in ok]

    print(f"{'reasoning_type':22} {'n':>4} {'accuracy':>9} {'formato':>8} {'cert. valid (su yes)':>21} {'giusta ma cert. non valido':>27}")
    for name, reps in groups.items():
        yes = [x for x in reps if x["llm_answer"] == "yes"]
        right_bad = [x for x in reps if x["answer_correct"] and x["llm_answer"] == "yes" and not x["certificate_valid"]]
        print(f"{name:22} {len(reps):4} {pct([x['answer_correct'] for x in reps]):>9} "
              f"{pct([x['format_consistent'] for x in reps]):>8} "
              f"{pct([bool(x['certificate_valid']) for x in yes]):>21} {len(right_bad):>27}")


def summarize_no_checker(path):
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(l) for l in f if l.strip()]
    print(f"{len(rows)} record in {path}")

    def parse_answer(raw):
        a = (raw or "").strip().strip('."\'`* ').lower()
        return a if a in ("yes", "no") else None

    reps = [{"llm_answer": parse_answer(r["raw_output"]),
             "ground_truth": r["dataset_answer"],
             "reasoning_type": r["reasoning_type"]} for r in rows]
    for x in reps:
        x["answer_correct"] = x["llm_answer"] == x["ground_truth"]
    print(f"  output non valido: {sum(x['llm_answer'] is None for x in reps)}\n")

    groups = {}
    for x in reps:
        groups.setdefault(x["reasoning_type"], []).append(x)
    groups["TOTALE"] = reps

    print(f"{'reasoning_type':22} {'n':>4} {'accuracy':>9} {'formato':>8} {'acc. su gt=yes':>15} {'acc. su gt=no':>14}")
    for name, g in groups.items():
        gt_yes = [x for x in g if x["ground_truth"] == "yes"]
        gt_no = [x for x in g if x["ground_truth"] == "no"]
        print(f"{name:22} {len(g):4} {pct([x['answer_correct'] for x in g]):>9} "
              f"{pct([x['llm_answer'] is not None for x in g]):>8} "
              f"{pct([x['answer_correct'] for x in gt_yes]):>15} {pct([x['answer_correct'] for x in gt_no]):>14}")



In [ ]:

print("=== qwen3.5:4b CON checker ===")
summarize_w_checker(QWEN_W_CHECKER_PATH)
print("\n=== qwen3.5:4b SENZA checker ===")
summarize_no_checker(QWEN_NO_CHECKER_PATH)

In [ ]:
# Confronto con / senza checker per qwen3.5:4b (stessa funzione compare usata per gemma4)
qwen_comparison_rows = compare(QWEN_W_CHECKER_PATH, QWEN_NO_CHECKER_PATH)

## Esperimento gemma4 con checker, temperature = 0

Stesso esperimento con checker di gemma4 (stesso prompt, schema, dataset, checker), cambia solo `temperature=0`.
Risultati in `gemma4_e4b_answers_w_checker_temp0.jsonl` (append, ripresa per `idx`): i file esistenti non vengono toccati.

Usa `run_dataset_model` / `run_one_w_checker_model` / `summarize_w_checker` definiti nella sezione qwen3.5:4b
(eseguire prima la cella di setup di quella sezione e i relativi prerequisiti).

In [ ]:
GEMMA_MODEL = "gemma4:e4b"
GEMMA_T0_PATH = f"{GEMMA_MODEL.replace(':', '_').replace('/', '_')}_answers_w_checker_temp0.jsonl"
assert GEMMA_T0_PATH not in ("gemma4_e4b_answers_w_checker.jsonl", "gemma4_e4b_answers_NO_checker.jsonl")

# Stesse opzioni del run gemma4 originale, tranne la temperatura
GEMMA_T0_GEN_OPTIONS = dict(
    num_ctx=32768,
    think=True,
    temperature=0,
    top_p=0.95,
    num_predict=-1,
    seed=42,
)

run_dataset_model(dataset, run_one_w_checker_model, GEMMA_MODEL, GEMMA_T0_PATH, GEMMA_T0_GEN_OPTIONS)

In [32]:
# Riepilogo: temperature 0 vs run originale (temperature 0.6)
GEMMA_MODEL = "gemma4:e4b"
GEMMA_T0_PATH = f"{GEMMA_MODEL.replace(':', '_').replace('/', '_')}_answers_w_checker_temp0.jsonl"
print("=== gemma4 con checker, temperature = 0 ===")
summarize_w_checker(GEMMA_T0_PATH)
print("\n=== gemma4 con checker, temperature = 0.6 (run originale) ===")
summarize_w_checker("gemma4_e4b_answers_w_checker.jsonl")

=== gemma4 con checker, temperature = 0 ===
152 record in gemma4_e4b_answers_w_checker_temp0.jsonl
  output non valido: 0 | errori checker: 0

reasoning_type            n  accuracy  formato  cert. valid (su yes)  giusta ma cert. non valido
subClassOf_direct        38     92.1%   100.0%                 94.3%                           2
subClassOf_inferred      38     84.2%   100.0%                 37.5%                          20
subClassOf_negative      76    100.0%   100.0%                    -                            0
TOTALE                  152     94.1%   100.0%                 67.2%                          22

=== gemma4 con checker, temperature = 0.6 (run originale) ===
152 record in gemma4_e4b_answers_w_checker.jsonl
  output non valido: 0 | errori checker: 0

reasoning_type            n  accuracy  formato  cert. valid (su yes)  giusta ma cert. non valido
subClassOf_direct        38     92.1%   100.0%                 88.6%                           4
subClassOf_inferred   